In [ ]:
# Pull the GCP credentials securely from Databricks Secrets
gcp_client_email = dbutils.secrets.get(scope="gcp_scope_gas_price_taxi_trips", key="client_email")
gcp_project_id = dbutils.secrets.get(scope="gcp_scope_gas_price_taxi_trips", key="project_id")
gcp_private_key = dbutils.secrets.get(scope="gcp_scope_gas_price_taxi_trips", key="private_key")
gcp_private_key_id = dbutils.secrets.get(scope="gcp_scope_gas_price_taxi_trips", key="private_key_id")

# Set them as a configuration.
sc = spark.sparkContext
hconf = sc._jsc.hadoopConfiguration()
hconf.set("google.cloud.auth.service.account.enable", "true")
hconf.set("fs.gs.project.id", gcp_project_id)
hconf.set("fs.gs.auth.service.account.email", gcp_client_email)
hconf.set("fs.gs.auth.service.account.private.key", gcp_private_key)
hconf.set("fs.gs.auth.service.account.private.key.id", gcp_private_key_id)

Load data from the GCP bucket 

In [ ]:
# Define GCP Paths
raw_taxi_file_path = "gs://taxi-data-group-project/taxi/yellow_tripdata_2025-10.parquet"
raw_gas_file_path = "gs://taxi-data-group-project/Gasoline_Retail_Prices_Weekly_Average_by_Region__Beginning_January_2017_20261120.csv"

rdd_file_path = "gs://taxi-data-group-project/rdds"

Create Taxi and Gas RDDs

In [ ]:
# Import data
taxi_rdd = spark.read.parquet(raw_taxi_file_path).rdd.map(lambda x: list(x))
gas_rdd = sc.textFile(raw_gas_file_path)\
    .map(lambda x : x.split(","))\
    .map(lambda row: [x.strip('"') for x in row])

Write the pre-processed data to the bucket

In [ ]:
taxi_rdd.saveAsTextFile(f'{rdd_file_path}/taxi')
gas_rdd.saveAsTextFile(f'{rdd_file_path}/gas')